# Lecture 18 — pandas: Asking Questions and Adding Columns
### EES 3350/5350 · Python in Earth Science · Friday, October 16, 2026

On Wednesday you met the DataFrame: columns by **name**, rows by **position** (`.iloc`) or **label** (`.loc`), and dates as labels. Today you'll do two things you already know how to do in NumPy, the pandas way:

1. **ask questions with masks**, and get back whole rows, not just one array,
2. **add new columns** to a table, and rename or remove old ones.

**Learning objectives for today**
- build a mask from a column, count it, and use it to filter **rows** of a DataFrame
- combine conditions with `&`, `|`, and `~` (same rules as Lecture 15)
- select rows by mask **and** a column by name in one step with `.loc[mask, "col"]`
- add a column calculated from other columns, and `.rename()` or `.drop()` columns

---
## Part 0 : get today's notebook

```
cd ~/course-materials
git pull
cp week08/18_pandas_masks_columns.ipynb ~/LastName_EES3350/lectures/18_pandas_masks_columns_LastName.ipynb
```

Then load the data with Wednesday's shortcut. We keep the columns we need and make our own `.copy()` (Lecture 15), because today we'll be **adding** columns.

In [ ]:
import numpy as np
import pandas as pd

raw = pd.read_csv("../../course-materials/week08/nashville_ghcnd_2025.csv", parse_dates=["DATE"], index_col="DATE")

weather = raw[["TMAX", "TMIN", "PRCP", "SNOW", "AWND"]].copy()

weather.head()

Five columns: daily high, daily low, precipitation, snowfall, and average wind speed, with the date as the label for every row.

*Data: NOAA NCEI GHCN-Daily, station USW00013897 (Nashville Intl AP, TN), 2025, via [Climate Data Online](https://www.ncdc.noaa.gov/cdo-web/), the same file as Lecture 17.*

---
## Part 1 : masks in pandas

In Lecture 15 you asked "on which days was something true?" with a **mask**. pandas uses the exact same idea, in three steps.

#### Step 1: make the mask
A comparison on a column checks every day and gives back a **boolean Series**: `True`/`False`, one per day, **labeled by date**.

In [ ]:
hot = weather["TMAX"] >= 90

hot.loc["2025-06-01":"2025-06-07"]

#### Step 2: count it
`True` counts as 1 and `False` as 0, so `.sum()` counts the hot days, the same as `np.sum(mask)`:

In [ ]:
print(f"Days at 90 F or hotter: {hot.sum()}")

79: the same answer as Lecture 15's `np.sum(tmax >= 90)`.

#### Step 3: filter with it
Put the mask inside square brackets on the **DataFrame**:

In [ ]:
hot_days = weather[hot]

print(hot_days.shape)
hot_days.head()

#### What just happened?
`weather[hot]` kept the **79 rows** where `hot` is `True`, and it kept **every column** for those rows: the high, the low, the precipitation, the snow, and the wind, all still lined up with their dates.

Compare that with NumPy, where every column was its own array:

|**question**|**NumPy (Lecture 15)**|**pandas (today)**|
|:--|:--|:--|
|which days were hot?|`tmax >= 90`|`weather["TMAX"] >= 90`|
|how many?|`np.sum(tmax >= 90)`|`(weather["TMAX"] >= 90).sum()`|
|the highs on those days|`tmax[tmax >= 90]`|`weather["TMAX"][weather["TMAX"] >= 90]`|
|**everything** on those days|`tmax[mask]`, `tmin[mask]`, `prcp[mask]`, ... one at a time|`weather[weather["TMAX"] >= 90]`, one table|

In one line, the usual pattern is:

In [ ]:
weather[weather["PRCP"] >= 1]

Read it out loud: **"the rows of `weather` where `PRCP` is at least 1."** That's the 12 days with an inch or more of rain from Lecture 15, and now you can see their dates and everything else about them.

#### Combining conditions: same rules as Lecture 15
`&` (both), `|` (either), `~` (not), with **each condition in its own parentheses**:

In [ ]:
hot_and_dry = weather[(weather["TMAX"] >= 90) & (weather["PRCP"] == 0)]

print(f"Hot, dry days: {len(hot_and_dry)}")

<div class="alert alert-danger">

**Common Error:** the Lecture 15 errors are back, with the same fixes:
- `(weather["TMAX"] >= 90) and (weather["PRCP"] == 0)` → `ValueError: The truth value of a Series is ambiguous`. Use `&`, not `and`.
- `weather["TMAX"] >= 90 & weather["PRCP"] == 0` → `TypeError: Cannot perform 'rand_' with a dtyped [float64] array and scalar of type [bool]`. A confusing message! Python did `90 & weather["PRCP"]` first. Add the parentheses.

pandas says "**Series** is ambiguous" where NumPy said "**array**," but it's the same mistake.

</div>

#### Rows by mask, column by name: `.loc[mask, "col"]`
Often you want the rows from a mask but only **one** column. `.loc` takes a row selector and a column selector, separated by a comma, and a mask counts as a row selector:

In [ ]:
print(f"Average wind on hot days: {weather.loc[hot, 'AWND'].mean():.1f} mph")
print(f"Average wind, all year:   {weather['AWND'].mean():.1f} mph")

<div class="alert alert-warning">

Hot days in Nashville were noticeably **calmer** than the year as a whole. 

</div>

|**you write**|**read it as**|**you get**|
|:--|:--|:--|
|`weather[mask]`|the rows where `mask` is `True`|a DataFrame, all columns|
|`weather.loc[mask, "AWND"]`|those rows, the `AWND` column|a Series|
|`weather.loc[mask, ["TMAX", "AWND"]]`|those rows, two columns|a DataFrame|
|`weather.loc["2025-07", "TMAX"]`|July (a label), the `TMAX` column|a Series (Wednesday)|

<div class="alert alert-info">

### Exercise 1 : asking the year questions

1. How many days had **any** precipitation (`PRCP` greater than 0) **and** a low at or below 32 °F? These are the days that *could* have had freezing rain or snow.
2. Make `big_rain`, the rows with **1 inch or more** of precipitation. What was the total precipitation on those days? (Lecture 15 answer: 24.66 in.)
3. Use `.loc` with a **mask and a column name** to find the average **high** on days with **1 inch or more** of precipitation. Is it warmer or cooler than the year's average high?
4. How many **summer** days (June 1 through August 31) reached **90 °F**? Hint: first make `summer` with `.loc` and dates (Wednesday), then use a mask on `summer`.

</div>

In [ ]:
# your code here
# 1
cold_wet = (weather["PRCP"] > 0) & (weather["TMIN"] <= ___)
print(f"Cold, wet days: {cold_wet.___}")

# 2
big_rain = weather[___]
print(f"Rain on 1-inch days: {big_rain[___].sum():.2f} in")

# 3
print(f"Average high on 1-inch days: {weather.loc[___, ___].mean():.1f} F")
print(f"Average high, all year:      {weather['TMAX'].mean():.1f} F")

# 4
summer = weather.loc[___:___]
print(f"Summer 90 F days: {(summer[___] >= 90).sum()}")

---
## Part 2 : adding, renaming, and removing columns

#### New columns from math
On Wednesday you noticed that NOAA's `TAVG` column was empty for this station, and in Lecture 14 we built our own daily average as `(tmax + tmin) / 2`. In pandas, assigning to a **new** column name adds that column to the table:

In [ ]:
weather["TAVG"] = (weather["TMAX"] + weather["TMIN"]) / 2

weather.head(3)

#### What just happened?
- On the right, `wx["TMAX"] + wx["TMIN"]` is element-wise math, day by day, exactly like arrays. pandas lines the days up by their **labels**.
- On the left, `wx["TAVG"] = ...` uses a name that doesn't exist yet, so pandas **adds** a column with that name at the end of the table.

The same pattern works for unit conversions:

In [ ]:
weather["TMAX_C"] = (weather["TMAX"] - 32) * 5 / 9
weather["PRCP_MM"] = weather["PRCP"] * 25.4

weather.head(3)

<div class="alert alert-success">

**Tip: name new columns with their units** (`TMAX_C`, `PRCP_MM`), so nobody, including you in two weeks, has to guess. If you write to a name that **already** exists (`wx["TMAX"] = ...`), pandas **replaces** that column. That's how raw data gets lost, so give converted values a new name instead.

</div>

#### Renaming columns
NOAA's codes are short, but `PRCP` and `AWND` aren't very readable. `.rename()` takes a **dictionary** (Lecture 05) of `{"old name": "new name"}` pairs:

In [ ]:
weather = weather.rename(columns={"PRCP": "PRCP_IN", "AWND": "WIND_MPH"})

weather.columns

Only the columns you list change; the rest keep their names. Notice the `wx = ...` at the start: `.rename()` gives back a **new** DataFrame, and the old one stays the same unless you save the result.

#### Removing columns
`.drop()` takes a list of column names:

In [ ]:
weather = weather.drop(columns=["SNOW"])

weather.columns

<div class="alert alert-danger">

**Common Error:** forgetting `weather = ...` in front of `.rename()` or `.drop()`. The cell runs with no error, prints the right-looking table, and then `weather` is **unchanged** the next time you use it. If a rename "didn't stick," check for the missing `weather =`.

</div>

<div class="alert alert-info">

### Exercise 2 : from scratch

No blanks this time.

1. Add a column called `RANGE`: the daily high minus the daily low.
2. On which date was the daily range the **biggest**, and how big was it? (Wednesday's `.idxmax()` gives you the date.)
3. Rename `RANGE` to `RANGE_F`, since it's in °F.
4. Use a mask to find the average daily range on **dry** days and on days with **any** precipitation. Which has the bigger range? Why might that be?

</div>

In [ ]:
# your code here

---
## Part 3 : a preview of Monday

Dates in the index know their own parts. `.month` pulls the month number out of every date, and it can become a column:

In [ ]:
weather["MONTH"] = weather.index.month

weather[["TMAX", "MONTH"]].head(3)

Now a mask can pick out any month:

In [ ]:
print(f"July average high: {weather[weather['MONTH'] == 7]['TMAX'].mean():.1f} F")

That's Lecture 15's 94.0 °F again. But what if you want the average high for **every** month? You'd write that line 12 times, or put it in a loop over `range(1, 13)`.

On Monday, you'll learn a way to do all 12 at once, in one line. It's called `groupby`, and it's the most useful thing in pandas.

<div class="alert alert-custom" style="background-color:#e8daef; border:1px solid #d2b4de; color:#6c3483; padding:10px; border-radius:4px;">

### Bonus: degree days, the pandas way

In the Lecture 14 bonus, you computed **cooling degree days** (how far each day's average temperature is **above** 65 °F, or 0 if it isn't) and **heating degree days** (how far it's **below**). Redo it with new columns:

1. Add `CDD` and `HDD` columns to `wx`, using `TAVG` and `np.maximum()` from Lecture 14. (`np.maximum()` works on a Series too.)
2. What were the year's totals? (Lecture 14 answers: 2,197 and about 3,034.)
3. Use a mask on `MONTH` to find which had more **cooling** degree days: **June** or **September**?

</div>

In [ ]:
# your code here

<details>
<summary style="color:green;"><strong>Answer.</strong></summary>

```python
# 1
weather["CDD"] = np.maximum(weather["TAVG"] - 65, 0)
weather["HDD"] = np.maximum(65 - weather["TAVG"], 0)

# 2
print(f"Cooling degree days: {weather['CDD'].sum():.0f}")
print(f"Heating degree days: {weather['HDD'].sum():.0f}")

# 3
print(f"June:      {weather.loc[weather['MONTH'] == 6, 'CDD'].sum():.0f}")
print(f"September: {weather.loc[weather['MONTH'] == 9, 'CDD'].sum():.0f}")
```
---
```
Cooling degree days: 2197
Heating degree days: 3034
June:      438
September: 320
```

For #3: June had more (438 vs. 320), even though September had 14 days at 90 °F or above. September's nights were cooler, so its daily averages ran lower.

NumPy functions like `np.maximum()` accept a Series and hand back a Series, with the dates still attached. That's what "pandas is built on NumPy" looks like in practice.

</details>

---
## A useful table

|**NumPy (Lecture 15)**|**pandas (today)**|
|:--|:--|
|`mask = tmax >= 90`|`mask = wx["TMAX"] >= 90`|
|`np.sum(mask)`|`mask.sum()`|
|`tmax[mask]`, then `tmin[mask]`, then ...|`wx[mask]`: every column at once|
|`(a >= 90) & (b == 0)`|`(wx["A"] >= 90) & (wx["B"] == 0)`: same rules|
|*(n/a)*|`wx.loc[mask, "AWND"]`: rows by mask, one column by name|
|`tavg = (tmax + tmin) / 2`, a new array|`wx["TAVG"] = (wx["TMAX"] + wx["TMIN"]) / 2`, a new column in the table|
|*(n/a)*|`wx = wx.rename(columns={"old": "new"})`|
|*(n/a)*|`wx = wx.drop(columns=["SNOW"])`|
|`month = data[:, 0]`|`wx["MONTH"] = wx.index.month`|